# Hello, Numerai

Hello and welcome to the Numerai Data Science Tournament!

This notebook is designed to help you build your first machine learning model and start competing in the tournament.

In [ ]:
!python --version
!pip install -q numerapi 'pandas>=2.3,<3' pyarrow lightgbm cloudpickle==3.1.1

## 1. Dataset

The Numerai dataset is a tabular dataset that describes the stock market over time, compiled from high-quality (and expensive) data that would be difficult for individuals to obtain.

## 2. Modeling
Our task: predict the `target` using the `features`.

### Model training

You are free to use any tool or framework, but here we use LightGBM - a popular choice amongst tournament participants. While your model trains, watch this [video](https://www.youtube.com/watch?v=w8Y7hY05z7k) from our Chief Scientist MDO on why tree-based models work so well on tabular data.

In [ ]:
from pathlib import Path
import json

import lightgbm as lgb
import pandas as pd
from numerapi import NumerAPI

DATA_VERSION = "v6.0"
napi = NumerAPI()
train_path = Path(DATA_VERSION) / "train.parquet"
features_path = Path(DATA_VERSION) / "features.json"

for path in (train_path, features_path):
    if not path.exists():
        napi.download_dataset(str(path))

with features_path.open() as file:
    feature_sets = json.load(file)["feature_sets"]
features = [name for name in feature_sets["small"] if name != "feature_country"]
assert features, "The small feature set has no numeric features"

train = pd.read_parquet(train_path, columns=features + ["target"])
train = train.dropna(subset=["target"])
assert not train.empty, "The training file has no mature target rows"
train = train.sample(n=min(200_000, len(train)), random_state=7)

model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    colsample_bytree=0.1,
    verbosity=-1,
)
model.fit(train[features], train["target"])
print(f"Trained on {len(train):,} rows and {len(features)} features")

## 3. Submissions

Unlike Kaggle competitions that evaluate models on <ins>test</ins> performance, Numerai evaluates models on <ins>live</ins> performance.

### Model upload

To automate this process, you can simply:
- Define your prediction pipeline as a function
- Serialize your function using the `cloudpickle` library
- Upload your model pickle file to Numerai
- Let Numerai run your model to submit live predictions every day

Read more about Model Uploads and other self-hosted automation options in our [docs](https://docs.numer.ai/numerai-tournament/submissions#automation).

In [ ]:
import cloudpickle


def predict(live_features: pd.DataFrame, live_benchmark_models: pd.DataFrame) -> pd.DataFrame:
    predictions = model.predict(live_features[features])
    ranked = pd.Series(predictions, index=live_features.index).rank(pct=True)
    return ranked.to_frame("prediction")


model_filename = "tutorial_basics.pkl"
with open(model_filename, "wb") as file:
    cloudpickle.dump(predict, file)
print(f"Saved {model_filename}")

### Live predictions

Your job is to predict the target: stock-specific returns 60 days into the future.

In [ ]:
live_path = Path(DATA_VERSION) / "live.parquet"
if not live_path.exists():
    napi.download_dataset(str(live_path))

live_features = pd.read_parquet(live_path, columns=features)
with open(model_filename, "rb") as file:
    saved_predict = cloudpickle.load(file)
example_predictions = saved_predict(live_features, pd.DataFrame(index=live_features.index))
assert example_predictions.index.equals(live_features.index)
assert example_predictions["prediction"].notna().all()
assert example_predictions["prediction"].between(0, 1).all()
example_predictions.head()

That's it! You now have a pickle file that is ready for upload.

Head back to the [Tournament Basics onboarding step](https://numer.ai/onboarding/tournament-basics) to upload your model!

In [ ]:
try:
    from google.colab import files
    files.download(model_filename)
except ImportError:
    print(f"Your model is saved locally as {model_filename}")